# 02 Data Ingestion

### confirm the files in the public S3 bucket

In [14]:
!aws s3 ls s3://ads508-team-4-talentiq-data/raw/

2026-03-16 02:59:14          0 
2026-03-16 03:04:34     381021 benefits.csv
2026-03-16 03:04:35    5887484 companies.csv
2026-03-16 03:04:35     439563 company_industries.csv
2026-03-16 03:04:36    3071450 company_specialities.csv
2026-03-16 03:04:36     895656 employee_attrition_data.csv
2026-03-16 03:04:36     490871 employee_counts.csv
2026-03-16 03:04:37     227977 ibm_hr.csv
2026-03-16 03:04:37     309470 job_industries.csv
2026-03-16 03:04:33   64179960 job_postings.csv
2026-03-16 03:04:38     427891 job_skills.csv


In [15]:
import boto3
import sagemaker
import pandas as pd

sess = sagemaker.Session()
bucket = "ads508-team-4-private"
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")

sm = boto3.Session().client(service_name="sagemaker", region_name=region)

In [16]:
print("Default bucket: {}".format(bucket))

Default bucket: ads508-team-4-private


### Set S3 Source Location and Destination Location

In [33]:
# set S3 source location and destination location
s3_public_path = "s3://ads508-team-4-talentiq-data/raw"
s3_private_path = "s3://ads508-team-4-private/raw"

!aws s3 ls {s3_public_path}/

# Copy Data From the Public S3 Bucket to our Private S3 Bucket in this Account

!aws s3 cp --recursive {s3_public_path}/ {s3_private_path}/ --exclude "*" --include "ibm_hr.csv"
!aws s3 cp --recursive {s3_public_path}/ {s3_private_path}/ --exclude "*" --include "employee_attrition_data.csv"

# confirm files have been copied
!aws s3 ls {s3_private_path}/

2026-03-16 02:59:14          0 
2026-03-16 03:04:34     381021 benefits.csv
2026-03-16 03:04:35    5887484 companies.csv
2026-03-16 03:04:35     439563 company_industries.csv
2026-03-16 03:04:36    3071450 company_specialities.csv
2026-03-16 03:04:36     895656 employee_attrition_data.csv
2026-03-16 03:04:36     490871 employee_counts.csv
2026-03-16 03:04:37     227977 ibm_hr.csv
2026-03-16 03:04:37     309470 job_industries.csv
2026-03-16 03:04:33   64179960 job_postings.csv
2026-03-16 03:04:38     427891 job_skills.csv
copy: s3://ads508-team-4-talentiq-data/raw/ibm_hr.csv to s3://ads508-team-4-private/raw/ibm_hr.csv
copy: s3://ads508-team-4-talentiq-data/raw/employee_attrition_data.csv to s3://ads508-team-4-private/raw/employee_attrition_data.csv
2026-03-16 03:54:15          0 
2026-03-16 03:59:44     895656 employee_attrition_data.csv
2026-03-16 03:59:43     227977 ibm_hr.csv


# Load data into Pandas dataframe

In [34]:
bucket = "ads508-team-4-private"

ibm_path = f"s3://{bucket}/raw/ibm_hr.csv"
attrition_path = f"s3://{bucket}/raw/employee_attrition_data.csv"

print(ibm_path)
print(attrition_path)

s3://ads508-team-4-private/raw/ibm_hr.csv
s3://ads508-team-4-private/raw/employee_attrition_data.csv


In [35]:
df_ibm = pd.read_csv(ibm_path)
df_attrition = pd.read_csv(attrition_path)

In [41]:
df_ibm.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [42]:
print(df_ibm.shape)

(1470, 35)


In [37]:
df_attrition.head()

,Employee_ID,Age,Gender,Marital_Status,Department,Job_Role,Job_Level,Monthly_Income,Hourly_Rate,Years_at_Company,...,Overtime,Project_Count,Average_Hours_Worked_Per_Week,Absenteeism,Work_Environment_Satisfaction,Relationship_with_Manager,Job_Involvement,Distance_From_Home,Number_of_Companies_Worked,Attrition
0,1,58,Male,Single,Finance,Manager,5,7332,81,24,...,No,9,48,16,4,1,1,49,3,No
1,2,48,Female,Divorced,HR,Assistant,4,6069,55,18,...,Yes,9,57,10,4,1,1,25,1,No
2,3,34,Female,Married,Marketing,Manager,4,11485,65,6,...,Yes,3,55,1,1,4,3,21,1,Yes
3,4,27,Female,Divorced,HR,Manager,4,18707,28,12,...,No,9,53,2,3,4,1,46,2,No
4,5,40,Male,Married,HR,Analyst,1,16398,92,3,...,No,1,54,11,1,1,1,43,4,No


In [43]:
print(df_attrition.shape)

(10000, 26)


### Standardize columns and concatenate files

In [44]:
# Rename df_attrition columns to match df_ibm where possible
df_attrition_renamed = df_attrition.rename(columns={
    "Employee_ID": "EmployeeNumber",
    "Age": "Age",
    "Gender": "Gender",
    "Marital_Status": "MaritalStatus",
    "Department": "Department",
    "Job_Role": "JobRole",
    "Job_Level": "JobLevel",
    "Monthly_Income": "MonthlyIncome",
    "Hourly_Rate": "HourlyRate",
    "Years_at_Company": "YearsAtCompany",
    "Years_in_Current_Role": "YearsInCurrentRole",
    "Years_Since_Last_Promotion": "YearsSinceLastPromotion",
    "Work_Life_Balance": "WorkLifeBalance",
    "Job_Satisfaction": "JobSatisfaction",
    "Performance_Rating": "PerformanceRating",
    "Training_Hours_Last_Year": "TrainingTimesLastYear",
    "Overtime": "OverTime",
    "Project_Count": "NumProjects",
    "Average_Hours_Worked_Per_Week": "AverageHoursWorked",
    "Absenteeism": "Absenteeism",
    "Work_Environment_Satisfaction": "EnvironmentSatisfaction",
    "Relationship_with_Manager": "RelationshipSatisfaction",
    "Job_Involvement": "JobInvolvement",
    "Distance_From_Home": "DistanceFromHome",
    "Number_of_Companies_Worked": "NumCompaniesWorked",
    "Attrition": "Attrition"
})

In [45]:
common_cols = df_ibm.columns.intersection(df_attrition_renamed.columns)

In [46]:
print(common_cols)

Index(['Age', 'Attrition', 'Department', 'DistanceFromHome', 'EmployeeNumber',
       'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus',
       'MonthlyIncome', 'NumCompaniesWorked', 'OverTime', 'PerformanceRating',
       'RelationshipSatisfaction', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion'],
      dtype='object')


In [47]:
# retain common columns
df_ibm_aligned = df_ibm[common_cols]
df_attrition_aligned = df_attrition_renamed[common_cols]

In [48]:
df_combined = pd.concat([df_ibm_aligned, df_attrition_aligned], ignore_index=True)

In [50]:
#examine combined df
print(df_combined.shape)
print(df_combined.head())
print(df_combined["Attrition"].value_counts())

(11470, 23)
   Age Attrition              Department  DistanceFromHome  EmployeeNumber  \
0   41       Yes                   Sales                 1               1   
1   49        No  Research & Development                 8               2   
2   37       Yes  Research & Development                 2               4   
3   33        No  Research & Development                 3               5   
4   27        No  Research & Development                 2               7   

   EnvironmentSatisfaction  Gender  HourlyRate  JobInvolvement  JobLevel  ...  \
0                        2  Female          94               3         2  ...   
1                        3    Male          61               2         2  ...   
2                        4    Male          92               2         1  ...   
3                        4  Female          56               3         1  ...   
4                        1    Male          40               3         1  ...   

  MonthlyIncome  NumCompaniesWor

### Save combined attrition data to bucket

In [51]:
bucket = "ads508-team-4-private"

output_path = f"s3://{bucket}/processed/combined_attrition_data.csv"

df_combined.to_csv(output_path, index=False)

In [52]:
!aws s3 ls s3://ads508-team-4-private/processed/

2026-03-16 04:14:28     956385 combined_attrition_data.csv
